<a href="https://colab.research.google.com/github/AliefReefandi/Grokking-Deep-Learning/blob/main/Chapter_11_Neural_Networks_That_Understand_Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11 — Neural Networks That Understand Language

## Tujuan Pembelajaran

Pada chapter ini dipelajari bagaimana neural network dapat digunakan untuk memahami data berupa bahasa atau teks.

Konsep utama yang dibahas:
- Natural Language Processing (NLP)
- Supervised NLP
- Representasi teks menjadi angka
- Sentiment classification
- Word embedding
- Embedding layer
- Negative sampling
- Word analogy

Pada akhir chapter, neural network digunakan untuk mempelajari hubungan antar kata sehingga kata yang memiliki karakteristik atau konteks yang mirip memiliki representasi vektor yang berdekatan.

## Natural Language Processing (NLP)

Natural Language Processing (NLP) adalah bidang yang berfokus pada bagaimana komputer dapat memproses dan memahami bahasa manusia.

Beberapa contoh tugas NLP:
- menentukan bagian dari kalimat,
- melakukan sentiment classification,
- mengenali nama orang atau tempat,
- menghubungkan kata atau bagian teks,
- memprediksi kata yang hilang berdasarkan konteks.

Pada Deep Learning, teks harus diubah menjadi bentuk numerik terlebih dahulu karena neural network bekerja dengan angka.

## Supervised NLP

Pada supervised learning, data input digunakan untuk menghasilkan target atau prediksi.

Pada NLP, input awalnya berupa teks sehingga teks harus dikonversi menjadi angka.

Alurnya:

Raw Text
   ↓
Representasi numerik
   ↓
Neural Network
   ↓
Prediction

Cara teks diubah menjadi angka sangat penting karena neural network belajar berdasarkan hubungan atau korelasi antara input dan output.

## Mengubah Kata Menjadi Angka

Salah satu pendekatan sederhana adalah membuat vocabulary yang berisi seluruh kata yang terdapat dalam dataset.

Setiap kata kemudian diberikan index.

Contoh:

"the"    → 0
"movie"  → 1
"good"   → 2
"bad"    → 3

Dengan cara tersebut, teks dapat diproses oleh neural network.

In [4]:
!wget -q https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/reviews.txt
!wget -q https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/labels.txt

print("Dataset berhasil didownload!")

Dataset berhasil didownload!


In [5]:
import sys

f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

f = open('labels.txt')
raw_labels = f.readlines()
f.close()

tokens = list(map(lambda x: set(x.split(" ")), raw_reviews))

vocab = set()

for sent in tokens:
    for word in sent:
        if len(word) > 0:
            vocab.add(word)

vocab = list(vocab)

word2index = {}

for i, word in enumerate(vocab):
    word2index[word] = i

input_dataset = list()

for sent in tokens:
    sent_indices = []

    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""

    input_dataset.append(list(set(sent_indices)))

target_dataset = list()

for label in raw_labels:
    if label == 'positive\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

print("Jumlah vocabulary :", len(vocab))
print("Jumlah review     :", len(input_dataset))

Jumlah vocabulary : 74074
Jumlah review     : 25000


## Embedding Layer

Embedding adalah representasi sebuah kata dalam bentuk vector angka.

Misalnya:

"good" → [0.2, 0.5, -0.1, 0.7]

"great" → [0.3, 0.4, -0.2, 0.6]

Kata yang memiliki karakteristik atau fungsi yang mirip diharapkan memiliki representasi vector yang juga mirip.

Pada chapter ini, embedding sebenarnya berasal dari nilai weights pada layer pertama neural network.

Embedding membuat proses lebih efisien karena neural network tidak perlu melakukan perkalian terhadap seluruh vocabulary. Network cukup mengambil baris weight yang sesuai dengan kata yang digunakan.

## Neural Network untuk Sentiment Classification

Arsitektur yang digunakan:

Input Words
     ↓
Embedding Layer
     ↓
Sigmoid
     ↓
Output
     ↓
Positive / Negative

Embedding layer berfungsi mengubah kumpulan kata menjadi representasi vector.

Kemudian vector tersebut digunakan oleh neural network untuk memprediksi sentiment dari review.

In [6]:
import numpy as np

np.random.seed(1)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

alpha, iterations = (0.01, 2)
hidden_size = 100

weights_0_1 = (
    0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
)

weights_1_2 = (
    0.2 * np.random.random((hidden_size, 1)) - 0.1
)

correct, total = (0, 0)

for iter in range(iterations):

    for i in range(len(input_dataset) - 1000):

        x, y = (input_dataset[i], target_dataset[i])

        layer_1 = sigmoid(
            np.sum(weights_0_1[x], axis=0)
        )

        layer_2 = sigmoid(
            np.dot(layer_1, weights_1_2)
        )

        layer_2_delta = layer_2 - y

        layer_1_delta = (
            layer_2_delta.dot(weights_1_2.T)
        )

        weights_0_1[x] -= (
            layer_1_delta * alpha
        )

        weights_1_2 -= (
            np.outer(layer_1, layer_2_delta) * alpha
        )

        if np.abs(layer_2_delta) < 0.5:
            correct += 1

        total += 1

        if i % 10 == 9:
            progress = str(i / float(len(input_dataset)))

            sys.stdout.write(
                '\rIter:' + str(iter) +
                ' Progress:' + progress[2:4] +
                '.' + progress[4:6] +
                '% Training Accuracy:' +
                str(correct / float(total)) + '%'
            )

    print()

Iter:0 Progress:95.99% Training Accuracy:0.8321666666666667%
Iter:1 Progress:95.99% Training Accuracy:0.8667708333333334%


In [7]:
correct, total = (0, 0)

for i in range(len(input_dataset) - 1000, len(input_dataset)):

    x = input_dataset[i]
    y = target_dataset[i]

    layer_1 = sigmoid(
        np.sum(weights_0_1[x], axis=0)
    )

    layer_2 = sigmoid(
        np.dot(layer_1, weights_1_2)
    )

    if np.abs(layer_2 - y) < 0.5:
        correct += 1

    total += 1

print("Test Accuracy:", correct / float(total))

Test Accuracy: 0.85


## Comparing Word Embeddings

Setiap baris pada weights_0_1 dapat dianggap sebagai embedding sebuah kata.

Jika dua kata memiliki nilai weight yang mirip, berarti keduanya memiliki karakteristik yang mirip menurut pola yang dipelajari network.

Contohnya, kata yang sama-sama memiliki hubungan kuat dengan sentiment positif dapat memiliki embedding yang berdekatan.

In [8]:
from collections import Counter
import math

def similar(target='good'):

    target_index = word2index[target]

    scores = Counter()

    for word, index in word2index.items():

        raw_difference = (
            weights_0_1[index]
            - weights_0_1[target_index]
        )

        squared_difference = (
            raw_difference * raw_difference
        )

        scores[word] = -math.sqrt(
            sum(squared_difference)
        )

    return scores.most_common(10)

print(similar('good'))

[('good', -0.0), ('documentary', -0.7286672846241051), ('technique', -0.7365473470470766), ('glued', -0.7394561105963836), ('discovers', -0.7423587639594672), ('error', -0.7425223226586051), ('matt', -0.7465103449987853), ('modern', -0.7468816549662153), ('mahatma', -0.7474382095261843), ('glover', -0.7495110217521411)]


Fungsi `similar()` membandingkan embedding sebuah kata dengan embedding kata lainnya.

Semakin kecil jarak antar-vector, semakin mirip representasi kedua kata tersebut.

## Filling in the Blank

Untuk mendapatkan embedding yang lebih kaya, buku mengubah tugas menjadi prediksi kata yang hilang.

Contoh:

"Mary had a little lamb whose _____ was white as snow."

Network harus memprediksi kata yang hilang berdasarkan kata-kata di sekitarnya.

Pendekatan ini memberikan signal yang lebih kaya karena network tidak hanya belajar apakah sebuah review positif atau negatif, tetapi juga belajar hubungan antar-kata dalam sebuah konteks.

## Negative Sampling

Jika vocabulary memiliki ribuan kata, memprediksi seluruh vocabulary setiap kali training akan membutuhkan banyak komputasi.

Negative sampling mengurangi jumlah output yang harus diproses.

Daripada membandingkan target dengan seluruh vocabulary, network hanya menggunakan:
- 1 kata target
- beberapa kata negatif yang dipilih secara acak

Pada contoh buku digunakan:

negative = 5

In [11]:
import sys
import random
import math
from collections import Counter
import numpy as np

np.random.seed(1)
random.seed(1)

f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

tokens = list(
    map(lambda x: x.split(" "), raw_reviews)
)

wordcnt = Counter()

for sent in tokens:
    for word in sent:
        wordcnt[word] -= 1

vocab = list(
    set(map(lambda x: x[0], wordcnt.most_common()))
)

word2index = {}

for i, word in enumerate(vocab):
    word2index[word] = i

concatenated = list()
input_dataset = list()

for sent in tokens:

    sent_indices = []

    for word in sent:
        try:
            sent_indices.append(word2index[word])
            concatenated.append(word2index[word])
        except:
            ""

    input_dataset.append(sent_indices)

concatenated = np.array(concatenated)

random.shuffle(input_dataset)

alpha, iterations = (0.05, 2)
hidden_size, window, negative = (50, 2, 5)

weights_0_1 = (
    (np.random.rand(len(vocab), hidden_size) - 0.5)
    * 0.2
)

weights_1_2 = (
    np.random.rand(len(vocab), hidden_size) * 0
)

layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1

## Meaning Is Derived from Loss

Salah satu konsep penting pada chapter ini adalah:

> Makna yang dipelajari oleh neural network sangat dipengaruhi oleh loss function dan target yang digunakan.

Ketika network dilatih untuk memprediksi sentiment, embedding cenderung mengelompokkan kata berdasarkan hubungan dengan sentiment.

Ketika network dilatih untuk memprediksi kata berdasarkan context, embedding cenderung mengelompokkan kata berdasarkan hubungan kemunculannya dalam bahasa.

Jadi, tujuan training menentukan informasi yang dipelajari oleh embedding.

## Word Analogy

Word embedding dapat menghasilkan hubungan antar kata dalam bentuk operasi vector.

Contoh terkenal:

king - man + woman ≈ queen

Ide dasarnya:

king
  -
man
  +
woman
  =
queen

Hubungan tersebut muncul karena embedding menyimpan pola statistik dari hubungan antar-kata.

In [13]:
def analogy(
    positive=['terrible', 'good'],
    negative=['bad']
):

    norms = np.sum(
        weights_0_1 * weights_0_1,
        axis=1
    )

    norms.resize(
        norms.shape[0],
        1
    )

    normed_weights = (
        weights_0_1 * norms
    )

    query_vect = np.zeros(
        len(weights_0_1[0])
    )

    for word in positive:
        query_vect += (
            normed_weights[
                word2index[word]
            ]
        )

    for word in negative:
        query_vect -= (
            normed_weights[
                word2index[word]
            ]
        )

    scores = Counter()

    for word, index in word2index.items():

        raw_difference = (
            weights_0_1[index]
            - query_vect
        )

        squared_difference = (
            raw_difference * raw_difference
        )

        scores[word] = -math.sqrt(
            sum(squared_difference)
        )

    return scores.most_common(10)[1:]

In [14]:
analogy(
    positive=['terrible', 'good'],
    negative=['bad']
)

[('decent', -187.6189027951816),
 ('nice', -187.64536712814453),
 ('great', -187.7218153333208),
 ('no', -187.80398216479654),
 ('perfect', -187.9276145919511),
 ('fine', -188.18329804304543),
 ('pretty', -188.18430144419403),
 ('up', -188.2264188412083),
 ('fun', -188.2534680574989)]

## Rangkuman

Pada Chapter 11 dipelajari bagaimana neural network digunakan untuk memproses bahasa.

Konsep utama:

1. NLP digunakan untuk memproses dan memahami bahasa manusia.
2. Teks harus diubah menjadi representasi numerik sebelum diproses neural network.
3. Word embedding merepresentasikan kata dalam bentuk vector.
4. Kata dengan karakteristik yang mirip dapat memiliki embedding yang mirip.
5. Embedding layer dapat mempercepat proses karena hanya memilih weight yang relevan.
6. Negative sampling digunakan untuk mengurangi jumlah output yang diproses.
7. Loss function memengaruhi jenis informasi yang dipelajari oleh embedding.
8. Word embedding dapat menghasilkan hubungan antar kata melalui operasi vector.
9. Salah satu contoh terkenal adalah:

   king - man + woman ≈ queen

## Kesimpulan

Chapter 11 memperkenalkan penerapan Deep Learning pada Natural Language Processing. Neural network tidak dapat langsung memahami teks sehingga kata harus direpresentasikan dalam bentuk numerik melalui word embedding.

Embedding yang dihasilkan tidak hanya menyimpan angka secara acak, tetapi mempelajari hubungan antar-kata berdasarkan tujuan dan loss function yang digunakan saat training. Konsep ini menjadi dasar penting untuk memahami bagaimana neural network dapat memproses bahasa dan menjadi pengantar menuju penggunaan Recurrent Neural Network pada Chapter 12.